# Data Engineering & Medallion Lakehouse Pipeline (Bronze → Silver → Gold)

**Consulting Engagement: BigDataCompany - Big Data Analytics Architecture**  
*Course: Big Data Analysis (BSc Labs) - NOVA IMS*  
*Client Prototype: Scalable Data Preparation Pipeline for MLlib & GraphFrames*

---

### Architectural Design: The Medallion Lakehouse
To transform raw, unstructured social media data into reliable, production-ready assets, this notebook implements the **Medallion Architecture** (Weeks 7–9 of the course curriculum):

```
Raw CSV Sources ──▶ [Bronze Layer: Raw Ingestion & Lineage]
                           │
                           ▼
                    [Silver Layer: Cleansed, Validated & Formatted]
                     ├── Silver Cleaned Text (ML-Ready + Indexed Labels)
                     ├── Silver Graph Vertices (id, followers, verified)
                     └── Silver Graph Edges (src, dst, timestamp)
                           │
                           ▼
                    [Gold Layer: Business-Level Analytical Marts]
                     ├── Gold Emotion Intelligence Metrics
                     └── Gold Network Centrality Features
```

Each stage is persisted in **Delta / Parquet format**, ensuring schema enforcement, fast columnar reads, and atomic writes.


In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, 
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.functions import (
    col, lower, regexp_replace, trim, when, current_timestamp, 
    input_file_name, to_timestamp, length, size, split, count, 
    avg as spark_avg, max as spark_max
)
from pyspark.ml.feature import StringIndexer

# Initialize Spark Session with Delta Lake / Parquet optimization
spark = SparkSession.builder \
    .appName("Medallion_DataEngineering_Pipeline") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Path resolver
def resolve_data_path(filename):
    candidates = [
        f"/FileStore/tables/{filename}",
        f"dbfs:/FileStore/{filename}",
        f"../Datasets/{filename}",
        f"Datasets/{filename}",
        filename
    ]
    for p in candidates:
        clean_p = p.replace("dbfs:", "")
        if os.path.exists(clean_p) or p.startswith("/FileStore") or p.startswith("dbfs:"):
            return p
    return f"Datasets/{filename}"

mental_health_raw_path = resolve_data_path("mental_health.csv")
twitter_graph_raw_path = resolve_data_path("twitter_graph.csv")

# Lakehouse Target Base Directory (Works in Databricks DBFS or local repository)
lakehouse_base = "/FileStore/lakehouse" if os.path.exists("/FileStore") else "../data/lakehouse"
os.makedirs(lakehouse_base, exist_ok=True)

print(f"Ingestion Sources:")
print(f" - Text:  {mental_health_raw_path}")
print(f" - Graph: {twitter_graph_raw_path}")
print(f"Lakehouse Target Base: {lakehouse_base}")


## Step 1: Bronze Layer — Raw Ingestion & Lineage Audit

The **Bronze Layer** stores data exactly as ingested from source systems, but with strict schema enforcement to prevent type distortion, plus technical audit metadata (`_ingested_at`, `_source_file`) for data governance.


In [ ]:
# Define Explicit Schemas
mental_health_schema = StructType([
    StructField("ID", LongType(), nullable=False),
    StructField("Text", StringType(), nullable=True),
    StructField("Emotion", StringType(), nullable=True)
])

twitter_graph_schema = StructType([
    StructField("Source", StringType(), nullable=True),
    StructField("Destination", StringType(), nullable=True),
    StructField("Conversation ID", StringType(), nullable=True),
    StructField("Relationship Date (UTC)", StringType(), nullable=True),
    StructField("Source Joined Date", StringType(), nullable=True),
    StructField("Source Followers", DoubleType(), nullable=True),
    StructField("Source Followed", DoubleType(), nullable=True),
    StructField("Source Verified", StringType(), nullable=True),
    StructField("Destination Joined Date", StringType(), nullable=True),
    StructField("Destination Followers", DoubleType(), nullable=True),
    StructField("Destination Followed", DoubleType(), nullable=True),
    StructField("Destination Verified", StringType(), nullable=True)
])

# Ingest Raw Data
raw_mental_health_df = spark.read.csv(
    mental_health_raw_path, 
    header=True, 
    schema=mental_health_schema, 
    multiLine=True, 
    escape='"'
)

raw_twitter_graph_df = spark.read.csv(
    twitter_graph_raw_path, 
    header=True, 
    schema=twitter_graph_schema
)

# Append Lineage Audit Metadata
bronze_mental_health = raw_mental_health_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", input_file_name())

bronze_twitter_graph = raw_twitter_graph_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", input_file_name())

# Helper to write as Delta (or Parquet fallback)
def write_table(df, folder_name, mode="overwrite"):
    target_path = f"{lakehouse_base}/{folder_name}"
    try:
        df.write.format("delta").mode(mode).save(target_path)
        print(f" Saved Delta table: {target_path}")
    except Exception as e:
        df.write.format("parquet").mode(mode).save(target_path)
        print(f" Saved Parquet table: {target_path}")

# Persist Bronze Tables
write_table(bronze_mental_health, "bronze/mental_health")
write_table(bronze_twitter_graph, "bronze/twitter_graph")

print(f"Bronze Mental Health Records: {bronze_mental_health.count():,}")
print(f"Bronze Twitter Graph Records: {bronze_twitter_graph.count():,}")


## Step 2: Silver Layer — Cleansing, Graph Structuring & Label Indexing

In the **Silver Layer**, raw data is sanitized, validated, and enriched:
1. **Text Cleansing:** Multi-stage Regex cleaning strips URLs, user mentions (`@user`), hashtag symbols (`#`), and non-alphabetic noise, followed by whitespace normalization.
2. **Label Encoding (`StringIndexer`):** The string column `Emotion` is converted into a numerical index `label` ($0.0$ to $5.0$) with class metadata preserved for Spark MLlib.
3. **GraphFrames Structure:** We separate the graph into:
   * **`silver_graph_edges`**: Required columns `src`, `dst`, relationship timestamp, conversation ID.
   * **`silver_graph_vertices`**: Required column `id`, plus deduplicated node attributes (`followers`, `following`, `verified`, `joined_date`).


In [ ]:
# 1. Advanced Text Cleansing Pipeline
clean_text_df = bronze_mental_health \
    .filter(col("Text").isNotNull() & (length(trim(col("Text"))) > 0)) \
    .withColumn("Cleaned_Text", regexp_replace(col("Text"), r"http\S+|www\.\S+", "")) \
    .withColumn("Cleaned_Text", regexp_replace(col("Cleaned_Text"), r"@\w+", "")) \
    .withColumn("Cleaned_Text", regexp_replace(col("Cleaned_Text"), r"#", "")) \
    .withColumn("Cleaned_Text", regexp_replace(col("Cleaned_Text"), r"[^a-zA-Z\s]", " ")) \
    .withColumn("Cleaned_Text", lower(col("Cleaned_Text"))) \
    .withColumn("Cleaned_Text", trim(regexp_replace(col("Cleaned_Text"), r"\s+", " "))) \
    .filter(length(col("Cleaned_Text")) > 2)

# 2. StringIndexer for MLlib Compatibility
label_indexer = StringIndexer(inputCol="Emotion", outputCol="label")
silver_mental_health = label_indexer.fit(clean_text_df).transform(clean_text_df)

# Feature metadata
silver_mental_health = silver_mental_health \
    .withColumn("Char_Length", length(col("Cleaned_Text"))) \
    .withColumn("Word_Count", size(split(col("Cleaned_Text"), r"\s+")))

# Persist Silver Text Table
write_table(silver_mental_health, "silver/mental_health")

print("=== Silver Text Sample (ML-Ready) ===")
silver_mental_health.select("ID", "Cleaned_Text", "Emotion", "label", "Word_Count").show(5, truncate=60)


In [ ]:
# 3. Graph Structuring for GraphFrames
clean_graph = bronze_twitter_graph \
    .filter(col("Source").isNotNull() & col("Destination").isNotNull()) \
    .withColumn("src", trim(col("Source"))) \
    .withColumn("dst", trim(col("Destination"))) \
    .filter(col("src") != col("dst")) \
    .withColumn("Relationship_Timestamp", to_timestamp(col("Relationship Date (UTC)"))) \
    .withColumn("Source_Verified", when(lower(col("`Source Verified`")) == "true", True).otherwise(False)) \
    .withColumn("Destination_Verified", when(lower(col("`Destination Verified`")) == "true", True).otherwise(False))

# Build Edges DataFrame (src, dst, attributes)
silver_edges = clean_graph.select(
    col("src"),
    col("dst"),
    col("Relationship_Timestamp"),
    col("Conversation ID").alias("conversation_id")
)

# Build Vertices DataFrame (id, followers, following, verified, joined_date)
src_vertices = clean_graph.select(
    col("src").alias("id"),
    col("`Source Followers`").alias("followers"),
    col("`Source Followed`").alias("following"),
    col("Source_Verified").alias("verified"),
    col("`Source Joined Date`").alias("joined_date")
)

dst_vertices = clean_graph.select(
    col("dst").alias("id"),
    col("`Destination Followers`").alias("followers"),
    col("`Destination Followed`").alias("following"),
    col("Destination_Verified").alias("verified"),
    col("`Destination Joined Date`").alias("joined_date")
)

# Union and Deduplicate Vertices by ID
silver_vertices = src_vertices.unionByName(dst_vertices) \
    .dropDuplicates(["id"]) \
    .filter(col("id").isNotNull() & (length(col("id")) > 0))

# Persist Silver Graph Tables
write_table(silver_edges, "silver/graph_edges")
write_table(silver_vertices, "silver/graph_vertices")

print(f"Silver Validated Edges:    {silver_edges.count():,}")
print(f"Silver Distinct Vertices: {silver_vertices.count():,}")


## Step 3: Gold Layer — Curated Business-Ready Aggregates

The **Gold Layer** contains pre-aggregated, highly performant tables tailored for executive dashboards, reporting, and high-level consulting decision support:
1. **`gold_emotion_intelligence`**: Strategic summary of emotional category volume, average length, and lexical density.
2. **`gold_user_network_features`**: Combined node-level centrality profile joining out-degree, in-degree, followers, and verification status.


In [ ]:
# 1. Gold Emotion Intelligence Mart
total_silver_texts = silver_mental_health.count()

gold_emotion_intelligence = silver_mental_health.groupBy("Emotion", "label").agg(
    count("*").alias("Total_Records"),
    spark_avg("Word_Count").alias("Avg_Word_Count"),
    spark_avg("Char_Length").alias("Avg_Char_Length"),
    spark_max("Word_Count").alias("Max_Word_Count")
).withColumn("Class_Share_Pct", (col("Total_Records") / total_silver_texts) * 100.0) \
 .orderBy("label")

write_table(gold_emotion_intelligence, "gold/emotion_intelligence")

print("=== Gold Emotion Intelligence Table ===")
gold_emotion_intelligence.show()

# 2. Gold User Network Centrality Feature Mart
out_deg = silver_edges.groupBy("src").agg(count("*").alias("out_degree"))
in_deg = silver_edges.groupBy("dst").agg(count("*").alias("in_degree"))

gold_user_network_features = silver_vertices \
    .join(out_deg, silver_vertices.id == out_deg.src, "left") \
    .join(in_deg, silver_vertices.id == in_deg.dst, "left") \
    .select(
        col("id").alias("account_id"),
        col("followers"),
        col("following"),
        col("verified"),
        when(col("out_degree").isNull(), 0).otherwise(col("out_degree")).alias("out_degree"),
        when(col("in_degree").isNull(), 0).otherwise(col("in_degree")).alias("in_degree")
    )

write_table(gold_user_network_features, "gold/user_network_features")

print("=== Gold User Network Features (Top Influencer Accounts) ===")
gold_user_network_features.orderBy(desc("in_degree"), desc("followers")).show(5)


## Step 4: Pipeline Lineage Audit & Downstream Hand-off

### Summary of Lakehouse Assets Built:
* **Bronze Tables:** `bronze/mental_health`, `bronze/twitter_graph`
* **Silver Tables:** `silver/mental_health` (ML-ready clean text + numerical labels), `silver/graph_edges`, `silver/graph_vertices`
* **Gold Marts:** `gold/emotion_intelligence`, `gold/user_network_features`

### Hand-off to Subsequent Course Deliverables:
1. **Machine Learning (`GX_MLlib.ipynb`):** Directly load `silver/mental_health` to fit TF-IDF / HashingTF + Classifier pipelines.
2. **Graph Analytics (`GX_GraphFrames.ipynb`):** Directly construct `GraphFrame(silver_vertices, silver_edges)` to execute PageRank, Connected Components, and Motif Analysis.
3. **Structured Streaming Bonus (`GX_Streaming.ipynb`):** Simulate real-time message stream scored against the Silver MLlib model.
